# Inspect GraphCast → CorrDiff Zarr

Loads the Zarr store produced by `pipeline.py` and summarizes:
- dimensions, coordinates, and global attrs
- `era5` / `wrf` groups: shape, dtype, channel lists, time range
- a couple of sample slices at a chosen time index

Data layout: both `era5` and `wrf` are indexed as `(time, channel, south_north, west_east)`, with 6-hourly timesteps inherited from the GraphCast rollout.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

ZARR_PATH = "/raid/apaudel/AURORA-CORRDIFF-ARTIFACTS/DATASETS/GRAPHCAST/ZARR_FOR_CORRDIFF/graphcast_2014_2015.zarr"

ds = xr.open_zarr(ZARR_PATH, consolidated=False)
ds

## 1. Top-level summary

In [ ]:
print("Dimensions:")
for d, n in ds.sizes.items():
    print(f"  {d:20s} = {n}")

print("\nCoordinates:")
for c in ds.coords:
    v = ds.coords[c]
    print(f"  {c:20s} dims={v.dims} dtype={v.dtype} shape={v.shape}")

print("\nData variables:")
for v in ds.data_vars:
    da = ds[v]
    print(f"  {v:20s} dims={da.dims} dtype={da.dtype} shape={da.shape}")

print("\nGlobal attrs:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

## 2. Time axis

Time is CF-encoded as `hours since <origin>`. Decode to datetimes and check the 6-hour cadence.

In [ ]:
t = ds["time"]
print("time attrs:", dict(t.attrs))

units = t.attrs.get("units", "")
if units.startswith("hours since "):
    origin = np.datetime64(units[len("hours since "):].strip(), "ns")
    times = origin + t.values.astype("timedelta64[h]")
else:
    times = t.values

print(f"n_timesteps : {len(times)}")
print(f"first       : {times[0]}")
print(f"last        : {times[-1]}")
if len(times) > 1:
    dts = np.diff(times).astype('timedelta64[h]').astype(int)
    print(f"unique dt(h): {np.unique(dts)}")

## 3. ERA5 and WRF "groups"

Not Zarr sub-groups — they're separate data variables sharing `(time, *_channel, south_north, west_east)`.

In [ ]:
def summarize(name):
    da = ds[name]
    print(f"=== {name} ===")
    print(f"  dims   : {da.dims}")
    print(f"  shape  : {da.shape}")
    print(f"  dtype  : {da.dtype}")
    ch_dim = [d for d in da.dims if d.endswith("_channel")]
    if ch_dim:
        ch = ds[ch_dim[0]].values
        print(f"  {ch_dim[0]} ({len(ch)}):")
        for i, c in enumerate(ch):
            print(f"    [{i:3d}] {c}")
    print()

if "era5" in ds: summarize("era5")
if "wrf"  in ds: summarize("wrf")

## 4. Validity masks

In [ ]:
for v in ("era5_valid", "wrf_valid"):
    if v in ds:
        da = ds[v]
        vals = da.values
        print(f"{v:12s} shape={da.shape} true={vals.sum()} false={(~vals).sum()}")

## 5. Static / normalization variables

In [ ]:
for v in ("era5_center", "era5_scale", "wrf_center", "wrf_scale"):
    if v in ds:
        da = ds[v]
        print(f"{v:12s} dims={da.dims} shape={da.shape} dtype={da.dtype}")
        print(f"             min={float(da.min()):.4g} max={float(da.max()):.4g}")

## 6. Plot a few fields at a selected time index

In [ ]:
TIME_INDEX = 0       # pick any index in [0, n_timesteps)
ERA5_CHANNELS_TO_PLOT = [0, 1]   # by index
WRF_CHANNELS_TO_PLOT  = [0]

sel_time = times[TIME_INDEX]
print(f"Selected time index {TIME_INDEX}: {sel_time}")

def plot_channel(group, ch_idx):
    da = ds[group].isel(time=TIME_INDEX, **{f"{group}_channel": ch_idx})
    arr = da.values
    ch_name = ds[f"{group}_channel"].values[ch_idx]
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(arr, origin="lower", cmap="viridis")
    ax.set_title(f"{group}[{ch_idx}] = {ch_name}  @  {sel_time}")
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()
    print(f"  shape={arr.shape} min={arr.min():.4g} max={arr.max():.4g} mean={arr.mean():.4g}")

if "era5" in ds:
    for i in ERA5_CHANNELS_TO_PLOT:
        plot_channel("era5", i)

if "wrf" in ds:
    for i in WRF_CHANNELS_TO_PLOT:
        plot_channel("wrf", i)

## 7. Sanity: pick a time and print a 3×3 patch for one channel

In [ ]:
ti, ci = 0, 0
patch = ds["era5"].isel(time=ti, era5_channel=ci,
                         south_north=slice(0, 3), west_east=slice(0, 3)).values
print(f"era5[t={ti}, ch={ci} -> {ds['era5_channel'].values[ci]}]")
print(patch)